In [3]:
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
from huggingface_hub import InferenceClient

In [ ]:
print("load some data and train...")

df = pd.read_csv("data/pubmedqa_clean.csv")
df = df.dropna(subset=["medical_text"])

vectorizer = TfidfVectorizer(stop_words="english", max_features=10000)
medical_vectors = vectorizer.fit_transform(df["medical_text"])

def retrieve_context(query, top_n=5):
    
    #find the most relevant medical records
    query_vector = vectorizer.transform([query])
    similarity_scores = cosine_similarity(query_vector, medical_vectors).flatten()
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    contexts = []
    for idx in top_indices:
        contexts.append(df.iloc[idx]["context_text"])
    
    return "\n\n---\n\n".join(contexts)

#set the AI (Generation)
HF_TOKEN = "YOUR_TOKEN_HERE" 
# Χρησιμοποιούμε το μοντέλο Zephyr
# Χρησιμοποιούμε ξανά το Mistral, αλλά τώρα με τον σωστό τρόπο (conversational)
client = InferenceClient("Qwen/Qwen2.5-7B-Instruct", token=HF_TOKEN)

load some data and train...


In [7]:
def answer_medical_question(query):
    if not query.strip():
        return "Παρακαλώ γράψε μια ιατρική ερώτηση."
        
    #  retrieve the relative context from the medical database (PubMed)
    context = retrieve_context(query)
    
    messages = [
        {
            "role": "system", 
            "content": "You are an expert medical AI assistant. Read the provided medical contexts and answer the user's question with a concise summary. If the answer is not in the context, say 'I don't have enough information'."
        },
        {
            "role": "user", 
            "content": f"Medical Contexts:\n{context}\n\nQuestion: {query}"
        }
    ]

    
    try:
        response = client.chat_completion(messages=messages, max_tokens=300, temperature=0.2)
        return response.choices[0].message.content
    except Exception as e:
        return f"Σφάλμα API: {str(e)}"
    
# create UI 
iface = gr.Interface(
    fn=answer_medical_question,
    inputs=gr.Textbox(lines=2, placeholder="Π.χ. Does aspirin reduce cardiovascular risk?"),
    outputs=gr.Textbox(lines=6, label="AI Περίληψη & Απάντηση"),
    title="⚕️ Ιατρικός RAG Βοηθός",
    description="Αυτό το σύστημα ψάχνει στη βάση ιατρικών δεδομένων (PubMed) και χρησιμοποιεί το AI (Mistral-7B) για να σου δώσει μια περιληπτική απάντηση.",
)

# lanch the app
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
